In [1]:
import zipfile
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. Đọc dữ liệu từ dataset.zip
zip_path = "../data/dataset.zip"
with zipfile.ZipFile(zip_path) as z:
    csv_filename = [f for f in z.namelist() if f.endswith(".csv")][0]
    with z.open(csv_filename) as f:
        df = pd.read_csv(f)

X = df.drop(columns=["Potability"])
y = df["Potability"]

# 2. Phân chia dữ liệu (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Tạo Pipeline kết hợp Tiền xử lý + Mô hình (Tránh Data Leakage)
# --- Mô hình 1: KNN ---
knn_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# --- Mô hình 2: Random Forest ---
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

# 4. Tinh chỉnh siêu tham số (GridSearchCV)
print("--- Đang huấn luyện và tinh chỉnh tham số KNN ---")
param_grid_knn = {'knn__n_neighbors': [3, 5, 7, 9, 11], 'knn__weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(knn_pipeline, param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)
grid_knn.fit(X_train, y_train)

print("--- Đang huấn luyện và tinh chỉnh tham số Random Forest ---")
param_grid_rf = {'rf__n_estimators': [50, 100, 200], 'rf__max_depth': [None, 10, 20]}
grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)
grid_rf.fit(X_train, y_train)

# 5. Đánh giá và Chọn mô hình tốt nhất
best_knn = grid_knn.best_estimator_
best_rf = grid_rf.best_estimator_

pred_knn = best_knn.predict(X_test)
pred_rf = best_rf.predict(X_test)

acc_knn = accuracy_score(y_test, pred_knn)
acc_rf = accuracy_score(y_test, pred_rf)

print(f"\n[Kết quả Test Set]")
print(f"- KNN Best Accuracy: {acc_knn:.4f} (Params: {grid_knn.best_params_})")
print(f"- Random Forest Best Accuracy: {acc_rf:.4f} (Params: {grid_rf.best_params_})")

# Chọn mô hình chiến thắng
if acc_rf >= acc_knn:
    best_model = best_rf
    best_model_name = "RandomForestClassifier"
    best_acc = acc_rf
    best_preds = pred_rf
else:
    best_model = best_knn
    best_model_name = "KNeighborsClassifier"
    best_acc = acc_knn
    best_preds = pred_knn

# 6. Lưu File Mô hình (model.joblib) vào ai-models/models/
model_path = "../models/model.joblib"
joblib.dump(best_model, model_path)
print(f"\n✅ Đã lưu file mô hình tại: {model_path}")

# 7. Lưu Schema (schema.json)
feature_names = list(X.columns)
schema = {
    "features": [
        {"name": col, "type": str(X[col].dtype), "required": True} for col in feature_names
    ],
    "target": {"name": "Potability", "type": "int64", "classes": [0, 1]}
}
schema_path = "../models/schema.json"
with open(schema_path, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4)
print(f"✅ Đã tạo file schema tại: {schema_path}")

# 8. Lưu Metadata (metadata.json)
metadata = {
    "model_name": best_model_name,
    "accuracy": round(float(best_acc), 4),
    "precision": round(float(precision_score(y_test, best_preds)), 4),
    "recall": round(float(recall_score(y_test, best_preds)), 4),
    "f1_score": round(float(f1_score(y_test, best_preds)), 4),
    "num_features": len(feature_names),
    "features_list": feature_names
}
metadata_path = "../models/metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)
print(f"✅ Đã tạo file metadata tại: {metadata_path}")

--- Đang huấn luyện và tinh chỉnh tham số KNN ---
--- Đang huấn luyện và tinh chỉnh tham số Random Forest ---

[Kết quả Test Set]
- KNN Best Accuracy: 0.6113 (Params: {'knn__n_neighbors': 11, 'knn__weights': 'uniform'})
- Random Forest Best Accuracy: 0.6540 (Params: {'rf__max_depth': 20, 'rf__n_estimators': 200})

✅ Đã lưu file mô hình tại: ../models/model.joblib
✅ Đã tạo file schema tại: ../models/schema.json
✅ Đã tạo file metadata tại: ../models/metadata.json
